# 🎯 RAGForge: Candidate Baseline Model Evaluation (Google Colab / Kaggle)

This notebook benchmarks pre-training candidate models against the **held-out RAG evaluation benchmark** (`rag_eval.jsonl`).

### Models Evaluated:
1. **`Qwen/Qwen2.5-1.5B-Instruct`** — Fast, compact conversational instruction model.
2. **`deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`** — Reasoning-specialized model with native `<think>` chain-of-thought traces.

### Core Metrics Recorded:
- **Concept Recall (%)**: Detection of essential architectural RAG mechanisms across all 6 taxonomy categories.
- **Reasoning Trace Analysis**: Presence and depth of step-by-step thinking traces.
- **Throughput & Latency**: Tokens/second and latency per question.
- **VRAM Footprint**: Memory usage on Tesla T4 (16 GB).

### Step 1: Check GPU Acceleration
Ensure your Colab runtime is set to **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies
We remove `torchvision` (not needed for text LLMs, prevents the Colab C++ `torchvision::nms` ABI collision) and install `transformers`, `accelerate`, and `tabulate`.

In [ ]:
!pip uninstall -y -q torchvision torchaudio
!pip install --upgrade -q transformers accelerate tabulate

> ⚠️ **Important:** If you see any operator error after running the cell above, click **Runtime -> Restart session** (or run the cell below to restart the session cleanly), then continue to Step 3.

In [ ]:
# Run this cell only if you need to restart the session after pip install
# import os; os.kill(os.getpid(), 9)

### Step 3: Upload or Load Benchmark Suite (`rag_eval.jsonl`)
Upload `rag_eval.jsonl` from your local `data/evaluation/` directory, or run the cell below to load.

In [ ]:
import os
from google.colab import files

if not os.path.exists("rag_eval.jsonl"):
    print("Please upload your data/evaluation/rag_eval.jsonl file:")
    uploaded = files.upload()
else:
    print("rag_eval.jsonl already present!")

### Step 4: Define Evaluation Engine & Metrics

In [ ]:
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SYSTEM_PROMPT = (
    "You are an expert RAG systems architect and researcher. "
    "Analyze the technical question carefully and provide a rigorous, grounded, and comprehensive explanation. "
    "Detail the underlying mechanisms, failure modes, trade-offs, and standard architectural solutions."
)

def normalize_term(term: str) -> str:
    term = term.lower()
    term = re.sub(r"[^\w\s-]", " ", term)
    return re.sub(r"\s+", " ", term).strip()

def extract_concept_variants(concept: str) -> List[str]:
    variants: Set[str] = set()
    parts = re.split(r"\s*/\s*|\s+or\s+", concept)
    for part in parts:
        cleaned = normalize_term(part)
        if cleaned:
            variants.add(cleaned)
        parens = re.findall(r"\(([^)]+)\)", part)
        for p in parens:
            p_clean = normalize_term(p)
            if p_clean:
                variants.add(p_clean)
        without_parens = normalize_term(re.sub(r"\([^)]*\)", "", part))
        if without_parens:
            variants.add(without_parens)

    hyphen_variants = set()
    for v in variants:
        if "-" in v:
            hyphen_variants.add(v.replace("-", " "))
            hyphen_variants.add(v.replace("-", ""))
    variants.update(hyphen_variants)
    return [v for v in variants if len(v) >= 2]

def score_concept_coverage(generated_text: str, expected_concepts: List[str]) -> Tuple[float, List[str], List[str]]:
    normalized_text = normalize_term(generated_text)
    spaced_text = normalized_text.replace("-", " ")
    matched = []
    missing = []
    for concept in expected_concepts:
        variants = extract_concept_variants(concept)
        is_matched = False
        for v in variants:
            if " " in v or len(v) >= 4:
                if v in normalized_text or v in spaced_text:
                    is_matched = True
                    break
            else:
                pattern = rf"\b{re.escape(v)}\b"
                if re.search(pattern, normalized_text) or re.search(pattern, spaced_text):
                    is_matched = True
                    break
        if is_matched:
            matched.append(concept)
        else:
            missing.append(concept)
    recall = len(matched) / len(expected_concepts) if expected_concepts else 0.0
    return round(recall, 4), matched, missing

def parse_reasoning_trace(text: str) -> Tuple[Optional[str], str]:
    think_match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
    if think_match:
        return think_match.group(1).strip(), re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return None, text.strip()

def evaluate_model(model_id: str, benchmark_path: str = "rag_eval.jsonl", max_new_tokens: int = 1024):
    print(f"\n{'='*70}\nEvaluating: {model_id}\n{'='*70}")
    with open(benchmark_path, "r", encoding="utf-8") as f:
        questions = [json.loads(line) for line in f if line.strip()]

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    results = []
    cat_recalls = {}
    total_toks = 0
    total_sec = 0.0

    for idx, item in enumerate(questions, 1):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["question"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        prompt_len = inputs["input_ids"].shape[1]

        t0 = time.perf_counter()
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.6,
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        t1 = time.perf_counter()

        duration = t1 - t0
        gen_ids = out_ids[0][prompt_len:]
        num_tokens = len(gen_ids)
        tok_sec = num_tokens / duration if duration > 0 else 0.0
        total_toks += num_tokens
        total_sec += duration

        decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)
        trace, answer = parse_reasoning_trace(decoded)
        recall, matched, missing = score_concept_coverage(decoded, item["expected_concepts"])
        cat_recalls.setdefault(item["category"], []).append(recall)

        results.append({
            "id": item["id"],
            "category": item["category"],
            "recall": recall,
            "tokens": num_tokens,
            "tok_per_sec": round(tok_sec, 2),
            "has_think": trace is not None,
            "matched": matched,
            "missing": missing,
        })
        print(f"  [{idx:02d}/{len(questions):02d}] {item['id']:<14} | Recall: {recall*100:5.1f}% | Speed: {tok_sec:5.1f} t/s | Cat: {item['category']}")

    avg_recall = sum(r["recall"] for r in results) / len(results)
    cat_summary = {k: round(sum(v)/len(v)*100, 1) for k, v in cat_recalls.items()}
    avg_speed = total_toks / total_sec if total_sec > 0 else 0.0

    report = {
        "model_id": model_id,
        "avg_recall_pct": round(avg_recall * 100, 2),
        "avg_speed_tok_s": round(avg_speed, 2),
        "category_recall_pct": cat_summary,
        "results": results,
    }
    del model
    torch.cuda.empty_cache()
    return report

### Step 5: Run Evaluation on Candidate 1 (`Qwen/Qwen2.5-1.5B-Instruct`)

In [ ]:
qwen_report = evaluate_model("Qwen/Qwen2.5-1.5B-Instruct")

### Step 6: Run Evaluation on Candidate 2 (`deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`)

In [ ]:
deepseek_report = evaluate_model("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")

### Step 7: Compare Results Side-by-Side

In [ ]:
from tabulate import tabulate

summary_rows = [
    ["Overall Concept Recall (%)", f"{qwen_report['avg_recall_pct']} %", f"{deepseek_report['avg_recall_pct']} %"],
    ["Average Throughput (tokens/s)", f"{qwen_report['avg_speed_tok_s']} t/s", f"{deepseek_report['avg_speed_tok_s']} t/s"],
]

for cat in qwen_report["category_recall_pct"]:
    q_score = qwen_report["category_recall_pct"].get(cat, 0.0)
    d_score = deepseek_report["category_recall_pct"].get(cat, 0.0)
    summary_rows.append([f"  - Category: {cat}", f"{q_score} %", f"{d_score} %"])

print(tabulate(summary_rows, headers=["Metric / Dimension", "Qwen 2.5 1.5B", "DeepSeek-R1-Distill-1.5B"], tablefmt="github"))

### Step 8: Save & Download Benchmark Reports

In [ ]:
with open("baseline_eval_qwen1_5b.json", "w", encoding="utf-8") as f:
    json.dump(qwen_report, f, indent=2)

with open("baseline_eval_deepseek_r1_1_5b.json", "w", encoding="utf-8") as f:
    json.dump(deepseek_report, f, indent=2)

files.download("baseline_eval_qwen1_5b.json")
files.download("baseline_eval_deepseek_r1_1_5b.json")
print("Downloaded evaluation reports successfully!")